In [13]:
from pymongo import MongoClient
import pandas as pd

client = MongoClient("127.0.0.1", 27017)
db = client["bridge"]

In [12]:
def statistics():
    data = []
    for lang in ["java", "py"]:
        tmp = []
        df = pd.DataFrame(
            db[f"{lang}_api_call_changes"].find({}, projection={"_id": 0})
        )
        num_records = len(df)
        num_commits = df["commit"].nunique()
        num_blobs = pd.concat([df["old_blob"], df["new_blob"]]).nunique()
        num_libraries = df["library"].nunique()
        num_versions = pd.concat(
            [
                df["library"] + " " + df["version_before"],
                df["library"] + " " + df["version_after"],
            ]
        ).nunique()

        lib_fqns = {}
        for row in df.itertuples(index=False):
            lib = row.library
            old_callees = row.old_callees
            new_callees = row.new_callees
            lib_fqns[lib] = lib_fqns.get(lib, set())
            for c in old_callees:
                lib_fqns[lib].add(c["full_name"])
            for c in new_callees:
                lib_fqns[lib].add(c["full_name"])
        num_fqns = sum(len(v) for v in lib_fqns.values())

        data.append(
            [num_records, num_commits, num_blobs, num_libraries, num_versions, num_fqns]
        )
    return pd.DataFrame(
        data,
        columns=[
            "# Records",
            "# Commits",
            "# Blobs",
            "# Libraries",
            "# Versions",
            "# API FQNs",
        ],
    )


statistics()

,# Records,# Commits,# Blobs,# Libraries,# Versions,# API FQNs
0,342018,42885,222243,5352,31084,138396
1,172999,31379,125861,2641,15540,36804
